# feature_kit — Usage Examples

This notebook walks through the core components of  using a synthetic dataset of 5 companies with 3 years of daily transaction history.

## 0. Generate Test Dataset

5 companies x 3 years (2021-01-01 to 2023-12-31) of daily observations.  
Metrics:  (daily revenue) and  (daily transaction count).

In [1]:
import polars as pl
import numpy as np
from datetime import date, timedelta

rng = np.random.default_rng(42)

companies = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]
start = date(2021, 1, 1)
n_days = 3 * 365
dates = [start + timedelta(days=i) for i in range(n_days)]

base_sales  = {"AAPL": 500, "MSFT": 400, "GOOGL": 350, "AMZN": 600, "META": 300}
base_count  = {"AAPL": 100, "MSFT":  80, "GOOGL":  70, "AMZN": 120, "META":  60}
growth_rate = {"AAPL": 0.10, "MSFT": 0.08, "GOOGL": 0.12, "AMZN": 0.15, "META": 0.05}

rows = []
for co in companies:
    for i, d in enumerate(dates):
        t = i / n_days
        trend = 1 + growth_rate[co] * t * 3
        sales = base_sales[co] * trend * (1 + 0.15 * rng.standard_normal())
        count = int(base_count[co] * trend * (1 + 0.10 * rng.standard_normal()))
        rows.append({"specid": co, "DATADATE": d, "SALES": round(float(sales), 2), "COUNT": float(max(count, 1))})

raw_df = pl.DataFrame(rows).sort(["specid", "DATADATE"])

print("Shape:", raw_df.shape)
print("Companies:", raw_df["specid"].unique().to_list())
print("Date range:", raw_df["DATADATE"].min(), "->", raw_df["DATADATE"].max())
raw_df.head(10)

Shape: (5475, 4)
Companies: ['AAPL', 'GOOGL', 'AMZN', 'META', 'MSFT']
Date range: 2021-01-01 -> 2023-12-31


specid,DATADATE,SALES,COUNT
str,date,f64,f64
"""AAPL""",2021-01-01,522.85,89.0
"""AAPL""",2021-01-02,556.44,109.0
"""AAPL""",2021-01-03,353.87,87.0
"""AAPL""",2021-01-04,510.01,96.0
"""AAPL""",2021-01-05,499.29,91.0
"""AAPL""",2021-01-06,566.73,107.0
"""AAPL""",2021-01-07,505.78,111.0
"""AAPL""",2021-01-08,536.09,91.0
"""AAPL""",2021-01-09,528.81,90.0


## 1. Params

In [2]:
from feature_kit import Param

params = Param(
    keys=["specid"],
    metrics=["SALES", "COUNT"],
)
print(params)

Param(keys=['specid'], metrics=['SALES', 'COUNT'], date_col='DATADATE')


## 2. Base Features

### 2a. Rolling Sum

In [3]:
from feature_kit import RollingSumBaseFeature

weekly_monthly_quarterly = [("weekly", 7), ("monthly", 28), ("quarterly", 91)]
rolling = RollingSumBaseFeature(params, windows=weekly_monthly_quarterly)
rolling_df = rolling.compute(raw_df)

print("Columns:", rolling_df.columns)
rolling_df.filter(pl.col("specid") == "AAPL").tail(5)

Columns: ['specid', 'DATADATE', 'SALES__ROLLING_SUM__WEEKLY', 'SALES__ROLLING_SUM__MONTHLY', 'SALES__ROLLING_SUM__QUARTERLY', 'COUNT__ROLLING_SUM__WEEKLY', 'COUNT__ROLLING_SUM__MONTHLY', 'COUNT__ROLLING_SUM__QUARTERLY']


specid,DATADATE,SALES__ROLLING_SUM__WEEKLY,SALES__ROLLING_SUM__MONTHLY,SALES__ROLLING_SUM__QUARTERLY,COUNT__ROLLING_SUM__WEEKLY,COUNT__ROLLING_SUM__MONTHLY,COUNT__ROLLING_SUM__QUARTERLY
str,date,f64,f64,f64,f64,f64,f64
"""AAPL""",2023-12-27,4703.33,18991.27,59491.85,871.0,3621.0,11626.0
"""AAPL""",2023-12-28,4594.67,18599.98,59366.06,866.0,3615.0,11624.0
"""AAPL""",2023-12-29,4508.82,18393.32,59167.46,881.0,3603.0,11624.0
"""AAPL""",2023-12-30,4527.07,18375.79,59302.34,891.0,3567.0,11606.0
"""AAPL""",2023-12-31,4691.44,18612.29,59519.84,898.0,3571.0,11630.0


### 2b. YoY Growth

In [4]:
from feature_kit import YOYGrowthBaseFeature

monthly_quarterly = [("monthly", 28), ("quarterly", 91)]
yoy = YOYGrowthBaseFeature(params, windows=monthly_quarterly)
yoy_df = yoy.compute(raw_df)

print("Columns:", yoy_df.columns)
yoy_df.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY', 'SALES__YOY_GROWTH__QUARTERLY', 'COUNT__YOY_GROWTH__MONTHLY', 'COUNT__YOY_GROWTH__QUARTERLY']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY,SALES__YOY_GROWTH__QUARTERLY,COUNT__YOY_GROWTH__MONTHLY,COUNT__YOY_GROWTH__QUARTERLY
str,date,f64,f64,f64,f64
"""AAPL""",2022-03-31,0.025988,0.094734,0.092758,0.104059
"""AAPL""",2022-04-01,0.030218,0.096673,0.085249,0.103236
"""AAPL""",2022-04-02,0.025622,0.097421,0.071504,0.1016
"""AAPL""",2022-04-03,0.039608,0.094398,0.072855,0.103331
"""AAPL""",2022-04-04,0.0652,0.099243,0.073391,0.101323


### 2c. Period-over-Period Growth

In [5]:
from feature_kit import POPGrowthBaseFeature

pop = POPGrowthBaseFeature(params, windows=monthly_quarterly)
pop_df = pop.compute(raw_df)

print("Columns:", pop_df.columns)
pop_df.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Columns: ['specid', 'DATADATE', 'SALES__POP_GROWTH__MONTHLY', 'COUNT__POP_GROWTH__MONTHLY', 'SALES__POP_GROWTH__QUARTERLY', 'COUNT__POP_GROWTH__QUARTERLY']


specid,DATADATE,SALES__POP_GROWTH__MONTHLY,COUNT__POP_GROWTH__MONTHLY,SALES__POP_GROWTH__QUARTERLY,COUNT__POP_GROWTH__QUARTERLY
str,date,f64,f64,f64,f64
"""AAPL""",2021-07-01,0.009556,0.036293,0.037944,0.03201
"""AAPL""",2021-07-02,0.01184,0.025113,0.039638,0.03006
"""AAPL""",2021-07-03,-0.005695,0.021997,0.038162,0.026633
"""AAPL""",2021-07-04,-0.02042,0.020536,0.034576,0.029038
"""AAPL""",2021-07-05,-0.029468,0.02253,0.039459,0.026451


## 3. Benchmarks

### 3a. Zero Benchmark

In [6]:
from feature_kit import ZeroBenchmark

zero_bench = ZeroBenchmark(params)
zero_df = zero_bench.compute(yoy_df, raw_df)
zero_df.head(5)

specid,DATADATE,ZERO_BENCHMARK
str,date,f64
"""AAPL""",2021-01-01,0.0
"""AAPL""",2021-01-02,0.0
"""AAPL""",2021-01-03,0.0
"""AAPL""",2021-01-04,0.0
"""AAPL""",2021-01-05,0.0


### 3b. Lagged YoY Growth Benchmark

The dev-branch API uses  and  to generate multiples of each period.

In [7]:
from feature_kit import LaggedYOYGrowthBenchmark

month_quarter = [("month", 28), ("quarter", 91)]
lagged_bench = LaggedYOYGrowthBenchmark(params, periods=month_quarter, num_lags=2)
lagged_df = lagged_bench.compute(yoy_df, raw_df)

print("Columns:", lagged_df.columns)
# LAG1MONTH = 28d ago, LAG2MONTH = 56d ago, LAG1QUARTER = 91d ago, LAG2QUARTER = 182d ago
lagged_df.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__LAG1MONTH', 'SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH', 'COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH', 'COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH', 'SALES__YOY_GROWTH__MONTHLY__LAG2MONTH', 'SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH', 'COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH', 'COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH', 'SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER', 'SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER', 'COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER', 'COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER', 'SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER', 'SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER', 'COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER', 'COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__LAG1MONTH,SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH,COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH,COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH,SALES__YOY_GROWTH__MONTHLY__LAG2MONTH,SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH,COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH,COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH,SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER,SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER,COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER,COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER,SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER,SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER,COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER,COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""AAPL""",2022-09-29,0.05418,0.07989,0.049246,0.08965,0.136899,0.07828,0.096074,0.117796,0.032602,0.076109,0.118327,0.114581,0.025988,0.094734,0.092758,0.104059
"""AAPL""",2022-09-30,0.047835,0.075031,0.056742,0.09362,0.1478,0.078333,0.093696,0.117299,0.035333,0.078362,0.122491,0.113857,0.030218,0.096673,0.085249,0.103236
"""AAPL""",2022-10-01,0.05185,0.076063,0.059409,0.09432,0.137859,0.076646,0.096214,0.117616,0.037358,0.078242,0.129826,0.119035,0.025622,0.097421,0.071504,0.1016
"""AAPL""",2022-10-02,0.038307,0.082104,0.05366,0.093408,0.13786,0.082352,0.097687,0.118008,0.060522,0.079289,0.126194,0.115643,0.039608,0.094398,0.072855,0.103331
"""AAPL""",2022-10-03,0.044483,0.083965,0.059409,0.093909,0.133424,0.084183,0.09244,0.115015,0.066739,0.076093,0.124068,0.116814,0.0652,0.099243,0.073391,0.101323


### 3c. Cohort Rolling Sum Benchmark

In [8]:
from feature_kit import CohortRollingSumBenchmark

cohort_raw = raw_df.with_columns(pl.lit("ALL").alias("cohort"))

rolling_for_cohort = RollingSumBaseFeature(params, windows=[("monthly", 28)]).compute(cohort_raw)
rolling_for_cohort = rolling_for_cohort.join(
    cohort_raw.select(["specid", "DATADATE", "cohort"]), on=["specid", "DATADATE"]
)

cohort_bench = CohortRollingSumBenchmark(params, cohort_cols=["cohort"], agg="mean")
cohort_df = cohort_bench.compute(rolling_for_cohort, cohort_raw)

print("Columns:", cohort_df.columns)
cohort_df.filter(pl.col("specid") == "AAPL").head(5)

Columns: ['specid', 'DATADATE', 'SALES__ROLLING_SUM__MONTHLY__COHORT_BENCHMARK', 'COUNT__ROLLING_SUM__MONTHLY__COHORT_BENCHMARK']


specid,DATADATE,SALES__ROLLING_SUM__MONTHLY__COHORT_BENCHMARK,COUNT__ROLLING_SUM__MONTHLY__COHORT_BENCHMARK
str,date,f64,f64
"""AAPL""",2021-01-01,null,null
"""AAPL""",2021-01-02,null,null
"""AAPL""",2021-01-03,null,null
"""AAPL""",2021-01-04,null,null
"""AAPL""",2021-01-05,null,null


## 4. Comparisons

### 4a. Difference & Sign

In [9]:
from feature_kit import DifferenceComparison, SignComparison

diff = DifferenceComparison().compute(yoy_df, zero_df)
sign = SignComparison().compute(yoy_df, zero_df)

print("Difference columns:", diff.columns)
diff.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Difference columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE
str,date,f64,f64,f64,f64
"""AAPL""",2022-03-31,0.025988,0.094734,0.092758,0.104059
"""AAPL""",2022-04-01,0.030218,0.096673,0.085249,0.103236
"""AAPL""",2022-04-02,0.025622,0.097421,0.071504,0.1016
"""AAPL""",2022-04-03,0.039608,0.094398,0.072855,0.103331
"""AAPL""",2022-04-04,0.0652,0.099243,0.073391,0.101323


In [10]:
print("Sign columns:", sign.columns)
sign.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Sign columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__SIGN', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN', 'COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__SIGN', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__SIGN,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__SIGN,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN
str,date,f64,f64,f64,f64
"""AAPL""",2022-03-31,1.0,1.0,1.0,1.0
"""AAPL""",2022-04-01,1.0,1.0,1.0,1.0
"""AAPL""",2022-04-02,1.0,1.0,1.0,1.0
"""AAPL""",2022-04-03,1.0,1.0,1.0,1.0
"""AAPL""",2022-04-04,1.0,1.0,1.0,1.0


### 4b. Rolling Z-Score

In [11]:
from feature_kit import ZScoreComparison

zscore = ZScoreComparison(params, lookback_window=91).compute(yoy_df, zero_df)

print("ZScore columns:", zscore.columns)
zscore.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

ZScore columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE', 'COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE']


specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE
str,date,f64,f64,f64,f64
"""AAPL""",2022-06-29,-0.650464,-0.928071,-0.076765,1.201949
"""AAPL""",2022-06-30,-0.809018,-0.954278,0.279369,1.556175
"""AAPL""",2022-07-01,-0.754993,-0.807748,0.428279,1.427722
"""AAPL""",2022-07-02,-0.717749,-0.799982,0.69933,2.034468
"""AAPL""",2022-07-03,-0.247503,-0.727257,0.538224,1.559915


### 4c. Cross-Sectional Rank

In [12]:
from feature_kit import RankComparison

rank = RankComparison(params, lookback_window=91).compute(yoy_df, zero_df)

rank_col = [c for c in rank.columns if "__RANK" in c][0]
print(f"Column: {rank_col}")
vmin = rank[rank_col].drop_nulls().min()
vmax = rank[rank_col].drop_nulls().max()
print(f"Range: [{vmin:.3f}, {vmax:.3f}]")


Column: SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__RANK
Range: [-0.978, 1.000]


### 4d. OLS Residual

In [13]:
from feature_kit import ResidualComparison

residual = ResidualComparison(params, lookback_window=91).compute(yoy_df, lagged_df)

print("Residual columns:", residual.columns)
residual.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

Residual columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__RESIDUAL', 'SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__RESIDUAL', 'SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__RESIDUAL', 'SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__RESIDUAL', 'SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__RESIDUAL', 'SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__RESIDUAL', 'COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH__RESIDUAL', 'COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH__RESIDUAL', 'COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER__RESIDUAL', 'COUNT__YOY_GROWTH__

specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__RESIDUAL,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__RESIDUAL,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER__RESIDUAL,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER__RESIDUAL,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER__RESIDUAL
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""AAPL""",2022-12-28,0.015737,0.032303,0.03958,0.030189,-0.01147,-0.010749,0.000749,-0.008642,-0.033502,-0.000676,0.007667,0.006477,-0.009707,0.019564,0.003375,0.008413
"""AAPL""",2022-12-29,0.021365,0.040779,0.043333,0.03897,-0.008356,-0.007563,0.003532,-0.005794,-0.022628,0.001942,0.022638,0.002065,-0.012008,0.016756,0.000723,0.00099
"""AAPL""",2022-12-30,0.041135,0.049447,0.046312,0.049247,0.00174,0.000267,0.009654,0.001823,-0.018742,0.000415,0.023644,-0.000335,-0.011064,0.020258,0.003928,0.002409
"""AAPL""",2022-12-31,0.051002,0.063382,0.055554,0.064044,0.005738,0.005369,0.012935,0.006677,-0.017165,0.007961,0.029157,-0.001381,-0.00886,0.016332,0.001633,-0.007586
"""AAPL""",2023-01-01,0.06135,0.085442,0.07277,0.085767,0.005021,0.005899,0.01246,0.007322,-0.020504,0.000461,0.023823,-0.001484,-0.008932,0.014629,0.001282,-0.001104


## 5. Transformations

### 5a. Winsorize

In [14]:
from feature_kit import WinsorizeTransformation

winsorized = WinsorizeTransformation(lower_q=0.01, upper_q=0.99).compute(diff)

feat_col = [c for c in diff.columns if "__VS__" in c][0]
b_min = diff[feat_col].drop_nulls().min()
b_max = diff[feat_col].drop_nulls().max()
a_min = winsorized[feat_col].drop_nulls().min()
a_max = winsorized[feat_col].drop_nulls().max()
print(f"Column: {feat_col}")
print(f"Before: min={b_min:.4f}, max={b_max:.4f}")
print(f"After:  min={a_min:.4f}, max={a_max:.4f}")

Column: SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE
Before: min=-0.0857, max=0.2504
After:  min=-0.0805, max=0.2433


### 5b. Ramp Up

Down-weight features for new entities in their first year.

In [15]:
from feature_kit import RampUpTransformation

age_lookup = raw_df.group_by("specid").agg(pl.col("DATADATE").min().alias("first_date"))
diff_with_age = (
    diff
    .join(age_lookup, on="specid")
    .with_columns(
        ((pl.col("DATADATE") - pl.col("first_date")).dt.total_days()).alias("age_days")
    )
    .drop("first_date")
)

ramped = RampUpTransformation(age_col="age_days", min_age=0, max_age=365).compute(diff_with_age)

feat_col = [c for c in diff.columns if "__VS__" in c][0]
ramped.select(["specid", "DATADATE", "age_days", feat_col]).filter(
    pl.col("specid") == "AAPL"
).drop_nulls().head(8)

specid,DATADATE,age_days,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE
str,date,i64,f64
"""AAPL""",2022-01-27,391,0.10114
"""AAPL""",2022-01-28,392,0.111064
"""AAPL""",2022-01-29,393,0.118189
"""AAPL""",2022-01-30,394,0.107189
"""AAPL""",2022-01-31,395,0.101003
"""AAPL""",2022-02-01,396,0.095237
"""AAPL""",2022-02-02,397,0.101273
"""AAPL""",2022-02-03,398,0.096638


## 6. Full Pipeline with FeatureBuilder

In [16]:
from feature_kit import (
    FeatureBuilder, Param,
    YOYGrowthBaseFeature,
    ZeroBenchmark, LaggedYOYGrowthBenchmark,
    DifferenceComparison, ZScoreComparison,
    WinsorizeTransformation,
)

params = Param(keys=["specid"], metrics=["SALES", "COUNT"])

builder = FeatureBuilder(
    params=params,
    base_feature=YOYGrowthBaseFeature(params, windows=[("monthly", 28), ("quarterly", 91)]),
    benchmarks=[
        ZeroBenchmark(params),
        LaggedYOYGrowthBenchmark(params, periods=[("month", 28), ("quarter", 91)], num_lags=2),
    ],
    comparisons=[
        DifferenceComparison(),
        ZScoreComparison(params, lookback_window=91),
    ],
    transformations=[
        WinsorizeTransformation(lower_q=0.01, upper_q=0.99),
    ],
)

result = builder.compute(raw_df)

print("Output shape:", result.shape)
print(f"\nOutput columns ({len(result.columns)} total):")
for col in result.columns:
    print(" ", col)

Output shape: (5475, 42)

Output columns (42 total):
  specid
  DATADATE
  SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE
  SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE
  COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE
  COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE
  SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE
  SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE
  COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE
  COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE
  SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE
  SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__DIFFERENCE
  SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__DIFFERENCE
  SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__DIFFERENCE
  SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE
  SALES__YOY_GROWTH

In [17]:
result.filter(pl.col("specid") == "AAPL").drop_nulls().head(5)

specid,DATADATE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER__ZSCORE
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""AAPL""",2022-12-28,0.127324,0.089036,0.124122,0.122753,0.907665,-1.248988,0.20043,1.511688,0.06007,0.040607,-0.041511,0.08644,-0.011992,-0.007903,-0.031317,0.012196,-0.026705,0.023309,0.033332,0.01548,0.010298,0.035394,0.036439,0.011074,1.074907,0.902452,-0.678275,1.021759,-0.627949,-0.673001,-2.366465,0.279419,-2.723367,-0.515357,0.029597,0.465882,0.032502,1.108453,1.472659,1.474069
"""AAPL""",2022-12-29,0.135738,0.09142,0.126736,0.120937,1.183296,-0.974788,0.315232,1.361676,0.076733,0.040998,-0.038232,0.100829,-0.010992,-0.009711,-0.032029,0.015311,-0.013241,0.026423,0.023307,0.009163,0.007192,0.034353,0.035346,0.006354,1.376943,0.908586,-0.596909,1.285737,-0.54776,-0.745384,-2.297

In [18]:
feat_cols = [c for c in result.columns if "__VS__" in c]
result.select(feat_cols).drop_nulls().describe()

statistic,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__ZERO_BENCHMARK__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__DIFFERENCE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER__DIFFERENCE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER__DIFFERENCE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1MONTH__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2MONTH__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG1QUARTER__ZSCORE,SALES__YOY_GROWTH__MONTHLY__VS__SALES__YOY_GROWTH__MONTHLY__LAG2QUARTER__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1MONTH__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2MONTH__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG1QUARTER__ZSCORE,SALES__YOY_GROWTH__QUARTERLY__VS__SALES__YOY_GROWTH__QUARTERLY__LAG2QUARTER__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1MONTH__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2MONTH__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG1QUARTER__ZSCORE,COUNT__YOY_GROWTH__MONTHLY__VS__COUNT__YOY_GROWTH__MONTHLY__LAG2QUARTER__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1MONTH__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2MONTH__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG1QUARTER__ZSCORE,COUNT__YOY_GROWTH__QUARTERLY__VS__COUNT__YOY_GROWTH__QUARTERLY__LAG2QUARTER__ZSCORE
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0,1845.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",0.089466,0.091434,0.084852,0.0865,-0.167563,-0.169292,-0.065602,-0.145161,-0.003297,-0.004335,-0.002782,-0.002677,-0.000881,-0.001188,-0.001587,-0.00095,-0.002764,-0.00271,-0.005188,-0.003915,-0.001595,-0.002429,-0.002724,-0.0

## 7. Preset Builders

In [19]:
from feature_kit.presets import sales_yoy_vs_zero, sales_rolling_vs_zero

params2 = Param(keys=["specid"], metrics=["SALES", "COUNT"])

yoy_result = sales_yoy_vs_zero(params2).compute(raw_df)
print("sales_yoy_vs_zero columns:", yoy_result.columns)

roll_result = sales_rolling_vs_zero(params2).compute(raw_df)
print("sales_rolling_vs_zero columns:", roll_result.columns)

sales_yoy_vs_zero columns: ['specid', 'DATADATE', 'SALES__YOY_GROWTH__QUADWEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__YOY_GROWTH__QUADWEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__YOY_GROWTH__QUADWEEKLY__VS__ZERO_BENCHMARK__SIGN', 'SALES__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN', 'COUNT__YOY_GROWTH__QUADWEEKLY__VS__ZERO_BENCHMARK__SIGN', 'COUNT__YOY_GROWTH__QUARTERLY__VS__ZERO_BENCHMARK__SIGN']
sales_rolling_vs_zero columns: ['specid', 'DATADATE', 'SALES__ROLLING_SUM__DAILY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__ROLLING_SUM__WEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__ROLLING_SUM__QUADWEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'SALES__ROLLING_SUM__QUARTERLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__ROLLING_SUM__DAILY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__ROLLING_SUM__WEEKLY__VS__ZERO_BENCHMARK__DIFFERENCE', 'COUNT__ROLLING_SUM__QUADWE